# SECOP II · Proyecto G002 en Google Colab

Ruta de trabajo: ingesta API → raw JSONL → PySpark y Parquet → calidad → SQLite → tablero interactivo. **Kafka, HDFS, MapReduce y streaming del proyecto integral se ejecutan en Docker**. Esta ruta de Colab no sustituye esas pruebas.


In [ ]:
!pip -q install requests pyspark==3.5.1 plotly


In [ ]:
import os,json,requests,time,datetime,pathlib
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
BASE="https://www.datos.gov.co/resource/jbjy-vk9h.json"
SES=requests.Session(); SES.mount("https://",HTTPAdapter(max_retries=Retry(total=5,backoff_factor=1,status_forcelist=[429,500,502,503,504])))
params={"$where":"fecha_de_firma IS NOT NULL AND id_contrato IS NOT NULL","$order":"fecha_de_firma DESC,id_contrato DESC"}
r=SES.get(BASE,params={"$select":"count(*)","$where":params["$where"]},timeout=60);r.raise_for_status();print("Disponibles:",r.json()[0]["count"])


In [ ]:
out=pathlib.Path("/content/secop_raw.jsonl");total=30000;count=0
with out.open("w",encoding="utf8") as f:
 while count<total:
  q={**params,"$limit":min(1000,total-count),"$offset":count}
  r=SES.get(BASE,params=q,timeout=120);r.raise_for_status();rows=r.json()
  if not rows: raise RuntimeError("Página vacía")
  for row in rows:
   rec={"fuente":"SECOP_II","dataset":"jbjy-vk9h","id_registro":str(row["id_contrato"]),"fecha_ingesta":datetime.datetime.now(datetime.timezone.utc).isoformat(),"payload":row}
   f.write(json.dumps(rec,ensure_ascii=False)+"\n")
  count+=len(rows)
print("Descargados:",count)


In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
spark=SparkSession.builder.master("local[*]").appName("SECOP_Colab").getOrCreate()
df=spark.read.json(str(out));print("Filas RAW:",df.count());df.select("id_registro","payload.fecha_de_firma","payload.valor_del_contrato").show(5,truncate=False)


In [ ]:
curada=(df.select("id_registro",F.to_date(F.substring("payload.fecha_de_firma",1,10)).alias("fecha"),F.col("payload.valor_del_contrato").cast("double").alias("valor"),F.col("payload.departamento").alias("departamento"))
.filter("id_registro IS NOT NULL AND fecha IS NOT NULL AND valor IS NOT NULL AND valor >= 0")
.dropDuplicates(["id_registro"]).withColumn("anio",F.year("fecha")))
print("Curados:",curada.count());curada.write.mode("overwrite").partitionBy("anio").parquet("/content/secop_parquet")
curada.groupBy("anio").agg(F.count("*").alias("contratos"),F.sum("valor").alias("valor_total")).orderBy("anio").show()


# Fase 6 · Calidad de datos e idempotencia
Estas pruebas operan sobre los 30.000 registros que descargaste. Se generan métricas verificables, un reporte y se comprueba que procesar dos veces los mismos registros no cambia el conteo de identificadores únicos. **La ingesta incremental y la prueba end-to-end de Kafka/HDFS se realizan en el proyecto Docker**, no quedan demostradas por esta celda.

In [ ]:
from pyspark.sql import functions as F
from pathlib import Path
import json, datetime

raw_rows = df.count()
null_id = df.filter(F.col("id_registro").isNull() | (F.trim("id_registro") == "")).count()
null_fecha = df.filter(F.to_date(F.substring("payload.fecha_de_firma", 1, 10)).isNull()).count()
valor_decimal = F.expr("try_cast(payload.valor_del_contrato as decimal(24,2))")
valor_invalido = df.filter(valor_decimal.isNull() | (valor_decimal < 0)).count()
unicos_raw = df.select("id_registro").distinct().count()
duplicados_raw = raw_rows - unicos_raw
curados = curada.count()
descartados = raw_rows - curados
id_nulos_curada = curada.filter(F.col("id_registro").isNull()).count()
duplicados_curada = curados - curada.select("id_registro").distinct().count()
fuera_rango = curada.filter(F.col("valor") < 0).count()
reporte = {
    "fecha_ejecucion_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "registros_raw": raw_rows, "registros_curados": curados,
    "descartados_o_deduplicados": descartados,
    "nulos_id_raw": null_id, "fechas_invalidas_raw": null_fecha,
    "valores_invalidos_raw": valor_invalido, "duplicados_raw": duplicados_raw,
    "nulos_id_curada": id_nulos_curada,
    "duplicados_curada": duplicados_curada,
    "valores_negativos_curada": fuera_rango,
}
assert curados > 0, "CALIDAD CRÍTICA: zona curada vacía"
assert id_nulos_curada == 0, "CALIDAD CRÍTICA: ID nulo"
assert duplicados_curada == 0, "CALIDAD CRÍTICA: ID duplicado"
assert fuera_rango == 0, "CALIDAD CRÍTICA: valor negativo"
Path('/content/reporte_calidad.json').write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(reporte, indent=2, ensure_ascii=False))

## Prueba reproducible de idempotencia lógica
Compara el conjunto original con una segunda copia exacta de los datos como si la ingesta se reintentara. La deduplicación por `id_registro` debe conservar el mismo total. Es una prueba de la **transformación batch**, no equivale a demostrar el reinicio de Kafka/streaming.

In [ ]:
segunda_ejecucion = df.unionByName(df)
antes_reintento = curada.count()
despues_reintento = (segunda_ejecucion.select(
    "id_registro",
    F.to_date(F.substring("payload.fecha_de_firma", 1, 10)).alias("fecha"),
    F.col("payload.valor_del_contrato").cast("double").alias("valor"),
    F.col("payload.departamento").alias("departamento")
).filter("id_registro IS NOT NULL AND fecha IS NOT NULL AND valor IS NOT NULL AND valor >= 0")
 .dropDuplicates(["id_registro"]).count())
print("Primera ejecución:", antes_reintento)
print("Dos cargas del mismo origen, tras deduplicación:", despues_reintento)
assert antes_reintento == despues_reintento, "No se conservó la idempotencia lógica"
print("PRUEBA CORRECTA: no se duplican IDs después de reprocesar")

# Fase 8 · Zona de servicio SQLite
Tres conjuntos de indicadores precalculados: resumen general, contratos por año y contratos por departamento. El tablero se alimenta exclusivamente de SQLite.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
DB='/content/secop_servicio.db'
con=sqlite3.connect(DB)
fecha_corte=datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
resumen=curada.agg(F.count('*').alias('total_contratos'),F.sum('valor').alias('valor_total'),F.countDistinct('departamento').alias('departamentos')).toPandas()
por_anio=curada.groupBy('anio').agg(F.count('*').alias('contratos'),F.sum('valor').alias('valor_total')).orderBy('anio').toPandas()
por_departamento=(curada.withColumn('departamento_normalizado',F.coalesce(F.upper(F.trim('departamento')),F.lit('SIN INFORMACIÓN')))
.groupBy('departamento_normalizado').agg(F.count('*').alias('contratos'),F.sum('valor').alias('valor_total'))
.orderBy(F.desc('valor_total')).toPandas())
resumen.to_sql('indicadores_generales',con,if_exists='replace',index=False)
por_anio.to_sql('indicadores_anuales',con,if_exists='replace',index=False)
por_departamento.to_sql('indicadores_departamentos',con,if_exists='replace',index=False)
con.execute('CREATE TABLE IF NOT EXISTS metadatos (clave TEXT PRIMARY KEY, valor TEXT)')
for k,v in {'fuente':'SECOP II / datos.gov.co - jbjy-vk9h','fecha_procesamiento':fecha_corte,'registros_raw':str(raw_rows),'registros_curados':str(curados)}.items():
 con.execute('INSERT OR REPLACE INTO metadatos (clave,valor) VALUES (?,?)',(k,v))
con.commit()
print(pd.read_sql_query('SELECT * FROM indicadores_generales',con))
print(pd.read_sql_query('SELECT * FROM indicadores_anuales ORDER BY anio',con))

## Tablero interactivo HTML
Se guardará en `/content/tablero_secop.html` y quedará disponible para descargar. **No se inventa ningún valor:** todos los indicadores provienen de las tablas SQLite.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

general=pd.read_sql_query('SELECT * FROM indicadores_generales',con).iloc[0]
anual=pd.read_sql_query('SELECT * FROM indicadores_anuales ORDER BY anio',con)
dpt=pd.read_sql_query('SELECT * FROM indicadores_departamentos ORDER BY valor_total DESC LIMIT 12',con)
fig=make_subplots(rows=2,cols=2,
 specs=[[{'type':'xy'},{'type':'xy'}],[{'type':'xy'},{'type':'domain'}]],
 subplot_titles=('Contratos por año','Valor contratado por año (billones COP)',
                  'Top departamentos por valor (billones COP)','Distribución por departamento'))
fig.add_trace(go.Bar(x=anual['anio'].astype(str),y=anual['contratos'],name='Contratos'),row=1,col=1)
fig.add_trace(go.Bar(x=anual['anio'].astype(str),y=anual['valor_total']/1e12,name='Valor anual'),row=1,col=2)
fig.add_trace(go.Bar(x=dpt['valor_total']/1e12,y=dpt['departamento_normalizado'],orientation='h',name='Departamentos'),row=2,col=1)
fig.add_trace(go.Pie(labels=dpt['departamento_normalizado'],values=dpt['valor_total'],hole=.45,name='Distribución'),row=2,col=2)
fig.update_layout(height=900,title=(f'¿Cómo se distribuyen los contratos públicos en Colombia?<br>'
 f'<sup>{int(general.total_contratos):,} contratos | Valor registrado: ${general.valor_total:,.0f} COP | '
 f'{int(general.departamentos)} departamentos | Fuente: SECOP II · Procesado: {fecha_corte}</sup>'),
 showlegend=False,template='plotly_white',margin=dict(l=130,r=30,t=115,b=40))
fig.write_html('/content/tablero_secop.html',include_plotlyjs=True,full_html=True)
fig.show()
print('Tablero guardado: /content/tablero_secop.html')

# Exportar evidencias y entregables de Colab
Descarga un ZIP con los archivos generados: calidad, base SQLite, dashboard HTML, ejemplo de raw y evidencia de conteos. Los archivos completos de infraestructura Kafka/HDFS y sus capturas se entregan desde Docker.

In [ ]:
import zipfile, pathlib
export='/content/evidencias_colab_secop.zip'
muestra=pathlib.Path('/content/muestra_raw.jsonl')
with open('/content/secop_raw.jsonl',encoding='utf-8') as src, muestra.open('w',encoding='utf-8') as dst:
 for _,line in zip(range(100),src):dst.write(line)
manifest={"fuente":"https://www.datos.gov.co/resource/jbjy-vk9h.json",
 "total_raw":raw_rows,"total_curado":curados,"fecha_procesamiento":fecha_corte,
 "observacion":"Manifest de evidencia de esta ejecución Colab; no reemplaza el manifiesto de paginación de la fase 2."}
pathlib.Path('/content/resumen_ejecucion.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
files=['/content/reporte_calidad.json','/content/secop_servicio.db','/content/tablero_secop.html',
       '/content/muestra_raw.jsonl','/content/resumen_ejecucion.json']
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in files:z.write(f,arcname=pathlib.Path(f).name)
print('Descargable:',export)
from google.colab import files as colab_files
colab_files.download(export)